# Sharing State with `multiprocessing.Manager`

https://docs.python.org/3/library/multiprocessing.html#managers

This notebook covers how to use `multiprocessing.Manager` objects to share more complex Python objects between processes. This provides a convenient way to manage shared state when simpler mechanisms like `Value` or `Array` are insufficient.

## The Problem: Sharing Complex Objects

As we know, processes have separate memory spaces. While `multiprocessing.Value` and `multiprocessing.Array` are great for sharing simple C-style data types (like numbers and basic arrays), they cannot handle rich Python objects like lists, dictionaries, or custom class instances.

This is where a **`Manager`** comes in. A manager object controls a **server process** which holds the actual Python objects. Other processes can then access and manipulate these shared objects using **proxies**.

### How Proxies Work

When you call `manager.list()`, you don't get a regular Python list. You get a **proxy object** that *behaves* like a list.

1.  Your process calls a method on the proxy (e.g., `shared_list.append(5)`).
2.  The proxy object internally pickles the arguments and sends a message to the manager's server process.
3.  The manager process receives the message, unpickles the arguments, and performs the `append(5)` operation on the *real* list it holds.
4.  Any return value is sent back to your process.

This mechanism makes sharing state incredibly convenient, but it comes with a **performance overhead** due to the inter-process communication for every operation.

---

## Sharing a List Between Processes

Let's see a simple example where multiple worker processes append their process ID to a shared list.

In [2]:
import multiprocessing
import time


In [3]:

def worker(shared_list):
    """
    Each worker appends its process ID into the shared list.
    """
    pid = multiprocessing.current_process().pid
    print(f"Worker {pid} appending to list")
    shared_list.append(pid)


if __name__ == "__main__":
    with multiprocessing.Manager() as manager:
        shared_list = manager.list()  # create a shared list
        processes = [multiprocessing.Process(target=worker, args=(shared_list,)) for _ in range(4)]

        for p in processes:
            p.start()
        for p in processes:
            p.join()

        print("Final shared list:", list(shared_list))


Worker 971 appending to list
Worker 973 appending to list
Worker 981 appending to list
Worker 989 appending to list
Final shared list: [971, 973, 981, 989]


As you can see, the main process has access to all the modifications made by the child processes. The order of items in the list may vary depending on the OS scheduler.

## Sharing a Dictionary


In [4]:
def worker_dict(shared_dict, key, value):
    shared_dict[key] = value


if __name__ == "__main__":
    with multiprocessing.Manager() as manager:
        shared_dict = manager.dict()
        processes = [
            multiprocessing.Process(target=worker_dict, args=(shared_dict, f"key{i}", i))
            for i in range(5)
        ]

        for p in processes:
            p.start()
        for p in processes:
            p.join()

        print("Final shared dict:", dict(shared_dict))


Final shared dict: {'key0': 0, 'key1': 1, 'key2': 2, 'key3': 3, 'key4': 4}


## Sharing a value counter (Lock to ensure safety)

In [5]:
def increment(shared_value, lock, n_times):
    for _ in range(n_times):
        with lock:  # ensure only one process updates at a time
            shared_value.value += 1


if __name__ == "__main__":
    with multiprocessing.Manager() as manager:
        shared_value = manager.Value('i', 0)  # 'i' means integer
        lock = manager.Lock()

        processes = [multiprocessing.Process(target=increment, args=(shared_value, lock, 1000)) for _ in range(5)]

        for p in processes:
            p.start()
        for p in processes:
            p.join()

        print("Final counter value:", shared_value.value)


Final counter value: 5000


## Shared Queue Example (Task Distribution)

In [6]:
def worker_queue(task_queue, result_queue):
    while not task_queue.empty():
        task = task_queue.get()
        result_queue.put(task ** 2)  # do some work


if __name__ == "__main__":
    with multiprocessing.Manager() as manager:
        task_queue = manager.Queue()
        result_queue = manager.Queue()

        # add tasks
        for i in range(10):
            task_queue.put(i)

        processes = [multiprocessing.Process(target=worker_queue, args=(task_queue, result_queue)) for _ in range(3)]

        for p in processes:
            p.start()
        for p in processes:
            p.join()

        results = [result_queue.get() for _ in range(result_queue.qsize())]
        print("Squared results:", results)


Squared results: [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


## Using Managers Server

In [6]:
#server.py
from multiprocessing.managers import BaseManager
import multiprocessing

# Shared objects
task_queue = multiprocessing.Queue()
result_queue = multiprocessing.Queue()

def get_task_queue():
    return task_queue

def get_result_queue():
    return result_queue


class QueueManager(BaseManager):
    pass


QueueManager.register('get_task_queue', callable=get_task_queue)
QueueManager.register('get_result_queue', callable=get_result_queue)


if __name__ == "__main__":
    print("[Server] Starting manager server...")
    manager = QueueManager(address=('127.0.0.1', 50000), authkey=b'secret')
    server = manager.get_server()
    server.serve_forever()   # <--- blocking call, runs server forever

[Server] Starting manager server...


SystemExit: 0

KeyboardInterrupt: 

In [ ]:
#client.py
from multiprocessing.managers import BaseManager
import multiprocessing

class QueueManager(BaseManager):
    pass

# Register proxies (no callables here)
QueueManager.register('get_task_queue')
QueueManager.register('get_result_queue')

def worker():
    manager = QueueManager(address=('127.0.0.1', 50000), authkey=b'secret')
    manager.connect()
    print(f"[Worker {multiprocessing.current_process().pid}] Connected to server")

    task_q = manager.get_task_queue()
    result_q = manager.get_result_queue()

    while not task_q.empty():
        num = task_q.get()
        print(f"[Worker {multiprocessing.current_process().pid}] got task {num}")
        result_q.put(num ** 2)

if __name__ == "__main__":
    worker()

In [ ]:
#feeder.py
from multiprocessing.managers import BaseManager

class QueueManager(BaseManager):
    pass

QueueManager.register('get_task_queue')
QueueManager.register('get_result_queue')

if __name__ == "__main__":
    manager = QueueManager(address=('127.0.0.1', 50000), authkey=b'secret')
    manager.connect()

    task_q = manager.get_task_queue()
    result_q = manager.get_result_queue()

    # feed tasks
    for i in range(5):
        task_q.put(i)
    print("[Feeder] Tasks added.")

    # wait for workers to finish
    import time; time.sleep(2)

    # collect results
    results = []
    while not result_q.empty():
        results.append(result_q.get())

    print("[Feeder] Results:", results)

---

## Race Conditions and Synchronization

While individual operations on a proxy object (like a single `append()` or `pop()`) are **atomic** (process-safe), more complex operations that involve multiple steps are **NOT**. This can easily lead to race conditions.

Consider a scenario where a worker needs to *read*, *modify*, and then *write back* a value. This is a classic "check-then-act" anti-pattern in concurrent programming.

Let's demonstrate this by having multiple processes increment a number stored inside our shared list.

In [ ]:
import multiprocessing

def faulty_incrementer(shared_list):
    """This function has a race condition."""
    for _ in range(10000):
        # READ -> MODIFY -> WRITE (not atomic!)
        current_value = shared_list[0]
        shared_list[0] = current_value + 1

manager = multiprocessing.Manager()
shared_list = manager.list([0]) # Start with a counter at 0

processes = [multiprocessing.Process(target=faulty_incrementer, args=(shared_list,)) for _ in range(4)]

for p in processes:
    p.start()

for p in processes:
    p.join()

# The expected value is 4 * 10000 = 40000
# The actual result will almost certainly be less.
print(f"Final counter value (with race condition): {shared_list[0]}")

### Solution: Using a Lock

To fix the race condition, we must ensure that the entire read-modify-write sequence is an atomic operation. We can do this by using a `Lock` to protect the critical section of code. You can create a lock from the same manager or from the `multiprocessing` module itself.


In [ ]:
import multiprocessing

def correct_incrementer(shared_list, lock):
    """This function is now process-safe."""
    for _ in range(10000):
        # Acquire the lock before the critical section
        with lock:
            # This block is now atomic
            current_value = shared_list[0]
            shared_list[0] = current_value + 1

manager = multiprocessing.Manager()
shared_list = manager.list([0])
lock = manager.Lock() # Create a lock from the manager

processes = [multiprocessing.Process(target=correct_incrementer, args=(shared_list, lock)) for _ in range(4)]

for p in processes:
    p.start()

for p in processes:
    p.join()

print(f"Final counter value (with lock): {shared_list[0]}")

---

## Summary and Best Practices

- **Convenience vs. Performance:** `Manager` objects are extremely convenient for sharing complex state but are slower than other IPC methods like `Queue` or `Array`/`Value` with a `Lock`. Use them when ease of development is more important than raw performance.

- **When to Use `Manager.list()`:** Ideal for situations where multiple processes need to read and write to a shared collection of data, and the data structure is more complex than a simple array of numbers.

- **Always Lock Compound Operations:** Remember that only single method calls on the proxy are atomic. Any sequence of operations that must be treated as a single unit **must be protected with a lock** to prevent race conditions.